# Reranker Export: cross-encoder/ms-marco-MiniLM-L-6-v2 → INT8 ONNX

Exports `cross-encoder/ms-marco-MiniLM-L-6-v2` to ONNX with INT8 quantization
and saves `reranker.onnx` + `reranker_tokenizer.json` under `modelserver/models/`.

**Run in the `training` uv group** which provides torch + transformers + optimum[onnxruntime]:
```
uv run --group training jupyter notebook notebooks/02_train_export_reranker.ipynb
```

After this notebook completes, update `modelserver/models/manifest.json` (T033) with
the reranker + reranker_tokenizer SHA-256 hashes printed at the end.

Image-budget check: `docker image inspect pantera-modelserver --format '{{.Size}}'`
must remain < 500 MB (Constitution VI). INT8 quantization of MiniLM-L-6 keeps the
model ~22 MB on disk.

In [ ]:
import hashlib
from pathlib import Path

MODEL_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"
OUT_DIR = Path("../modelserver/models")
RERANKER_ONNX = OUT_DIR / "reranker.onnx"
RERANKER_TOK = OUT_DIR / "reranker_tokenizer.json"

print(f"Output dir: {OUT_DIR.resolve()}")

## 1. Export to ONNX via Optimum

Uses `ORTModelForSequenceClassification` which exports the full cross-encoder
graph including all three BERT inputs (input_ids, attention_mask, token_type_ids).

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

# Export to a temp dir first, then quantize
import tempfile
tmp = Path(tempfile.mkdtemp())

print(f"Exporting {MODEL_ID} to ONNX...")
model = ORTModelForSequenceClassification.from_pretrained(
    MODEL_ID, export=True
)
model.save_pretrained(tmp)
print(f"FP32 ONNX saved to {tmp}")

onnx_fp32 = tmp / "model.onnx"
print(f"FP32 size: {onnx_fp32.stat().st_size / 1e6:.1f} MB")

## 2. INT8 Static Quantization

Quantizes with AVX2 operator set for CPU inference. Reduces ~90 MB FP32 → ~22 MB INT8.

In [ ]:
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from optimum.onnxruntime import ORTQuantizer

quantizer = ORTQuantizer.from_pretrained(tmp)
dqconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)

print("Quantizing to INT8 (dynamic)...")
quantizer.quantize(
    save_dir=tmp,
    quantization_config=dqconfig,
)

onnx_int8 = tmp / "model_quantized.onnx"
print(f"INT8 size: {onnx_int8.stat().st_size / 1e6:.1f} MB")

## 3. Verify ONNX inputs match CrossEncoderSession expectations

The `CrossEncoderSession` in `modelserver/inference/reranker.py` runs:
```python
session.run(None, {"input_ids": …, "attention_mask": …, "token_type_ids": …})
```
This cell verifies those three input names exist in the exported graph.

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(str(onnx_int8), providers=["CPUExecutionProvider"])
input_names = {inp.name for inp in sess.get_inputs()}
print("ONNX input names:", input_names)

required = {"input_ids", "attention_mask", "token_type_ids"}
assert required.issubset(input_names), (
    f"Missing inputs: {required - input_names}. "
    "Update CrossEncoderSession.rerank() to match actual input names."
)
print("All required inputs present. ✓")

## 4. Smoke-test the quantized model

In [ ]:
import numpy as np
from tokenizers import Tokenizer

# Load the HF tokenizer as fast tokenizer JSON for the modelserver
tok = AutoTokenizer.from_pretrained(MODEL_ID)
fast_tok = tok.backend_tokenizer

# Quick functional test
pairs = [
    ("hepatotoxicity associated with drug X", "patient developed severe liver damage"),
    ("hepatotoxicity associated with drug X", "the weather is nice today"),
]
encs = fast_tok.encode_batch(pairs)
input_ids = np.array([e.ids for e in encs], dtype=np.int64)
attn_mask = np.array([e.attention_mask for e in encs], dtype=np.int64)
type_ids = np.array([e.type_ids for e in encs], dtype=np.int64)

out = sess.run(None, {
    "input_ids": input_ids,
    "attention_mask": attn_mask,
    "token_type_ids": type_ids,
})
logits = out[0][:, 0]
print(f"Logits: {logits}")
assert logits[0] > logits[1], "Expected relevant passage to score higher"
print("Sanity check passed: relevant passage scores higher than irrelevant. ✓")

## 5. Copy artifacts and save tokenizer JSON

In [ ]:
import shutil

shutil.copy(onnx_int8, RERANKER_ONNX)
fast_tok.save(str(RERANKER_TOK))

print(f"reranker.onnx       → {RERANKER_ONNX} ({RERANKER_ONNX.stat().st_size / 1e6:.1f} MB)")
print(f"reranker_tokenizer.json → {RERANKER_TOK} ({RERANKER_TOK.stat().st_size / 1024:.1f} KB)")

## 6. Print SHA-256 hashes for manifest.json (T033)

In [ ]:
def sha256(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()

rr_sha = sha256(RERANKER_ONNX)
rr_tok_sha = sha256(RERANKER_TOK)

print("Add these entries to modelserver/models/manifest.json:")
print()
print('    {')
print('      "name": "reranker",')
print('      "file": "reranker.onnx",')
print('      "format": "onnx",')
print('      "version": "v1.0-ms-marco-minilm-l6-int8",')
print(f'      "sha256": "{rr_sha}",')
print('      "max_tokens": 512')
print('    },')
print('    {')
print('      "name": "reranker_tokenizer",')
print('      "file": "reranker_tokenizer.json",')
print('      "format": "tokenizer",')
print('      "version": "v1.0-ms-marco-minilm-l6",')
print(f'      "sha256": "{rr_tok_sha}"')
print('    }')